# Nightingale: the language-model baselines B3 and B4 and the grounded explainer (EXP-009, EXP-010), on Kaggle

This runs a small local language model, **`llama3.1:8b` through Ollama** (decision D-5), over a
sample, **on Kaggle's GPU and never on the laptop** (D-7, D-9). Everything it measures was
pre-registered in `docs/08` (EXP-009 and EXP-010) before it ever ran. Nothing is trained, and **the
test split is never downloaded**.

It makes 416 calls over the four golden cases
and 50 validation patients drawn by hash order (seed 42), at temperature 0:

| Arm | What it is | Calls |
|---|---|---|
| **B3** | the model alone: the case in the prompt, no retrieval, no graph, no ML ranker, at 100%, 50% and 25% of the history | 154 |
| **B3r** | B3 told to use only the facts it is given (the control that keeps H3 honest) | 54 |
| **B4** | text retrieval and the model: the best passage for each of the 14 conditions | 154 |
| **A0+LLM** | the pipeline as configured, with the language-model explainer on; the raw answer is kept beside what the guardrails let through | 54 |

Then it scores the run (`scripts/score_llm_baselines.py`): the pre-registered figures, H3's four
comparisons, **D-5's checkpoint of 2026-11-03**, and the sheet for the two team raters.

## What to add as input first (about 10 minutes of your time, once)

1. **`nightingale-models`**, your private dataset from the EXP-021 reports run (the three model
   bundles). Needed by A0+LLM only. *Add Input → Your Datasets.*
2. **The retrieval bundle**, the output of `notebooks/kaggle_retrieval_corpus.ipynb` once that run has
   **finished** (its `STATUS.txt` starts with `DONE`). Needed by B4 and A0+LLM. *Add Input → Your
   Work → Notebook Output Files → the corpus notebook's last version.* **Without it B3 and B3r still
   run**, and `STATUS.txt` says what was left out; you can run the notebook again later with the
   bundle added, and it resumes (below).
3. *Optional:* a private dataset holding `triples.jsonl` and `nl_facts.jsonl` from `data/raw/bodhi_s/`
   on the laptop (BODHI-S is CC-BY-NC and is never committed). It lets the graph here match the
   laptop's. Without it the graph is DDXPlus and the hand-authored facts only, and `run.json` says so.

## How to run it

1. On kaggle.com: **Create → New Notebook**, then **File → Import Notebook** and upload this file.
2. **Add Input** as above. In **Session options**:
   - **Accelerator: GPU T4 x2.** The model must run on a GPU: on the CPU it would take days, and the
     notebook stops if it looks like it is on one.
   - **Internet: on** (it clones the repository, downloads DDXPlus's validation split, installs Ollama
     from ollama.com and pulls the model, about 5 GB, from Ollama's registry).
3. Top right: **Save Version → Save & Run All (Commit) → Save.** **You can close the browser now.**
4. Come back in about **1–3 hours**. That is an estimate: nothing has run on Kaggle yet, and
   installing Ollama there has not been tried. The **smoke test** (the cell after the Ollama install) is the check: it runs
   two golden cases and two patients through every arm first, and a failure stops the run there, in
   a few minutes, before the long part.
5. Open the notebook's latest version → **Output**, download **`llm_explain.zip`**, unzip it on the
   laptop into `data/interim/llm_explain/` and tell Claude. **It holds corpus excerpts and DDXPlus
   patients, so it stays under `data/` and is never committed.**

**If the run stops early** (a Kaggle limit, an error): open the notebook, **Add Input → Your Work →
Notebook Output Files**, choose this notebook's last version, then **Save & Run All** again. The
calls already made are copied from that output and skipped. The same works to add B4 and A0+LLM
later: add the finished bundle as an input and run again.

**Keep the notebook private** (Kaggle's default). Nothing here needs a password or a token. The
model is Meta's Llama 3.1 and is used under its licence for this university project.

In [ ]:
# The commit Claude gave you. "master" works too: run.json records the exact commit.
COMMIT = "master"
MODEL = "llama3.1:8b"  # D-5
ARMS = ("B3", "B3r", "B4", "A0+LLM")  # B4 and A0+LLM are skipped, and said so, without the bundle
# Empty: the install script's latest, which run.json records. To repeat a run exactly, put the
# version it recorded here.
OLLAMA_VERSION = ""

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

REPO = Path("/tmp/nightingale")  # outside /kaggle/working, so no data ends up in the output
OUT = Path("/kaggle/working/llm_explain")
LOGS = OUT / "logs"
INPUT = Path("/kaggle/input")
for folder in (OUT, LOGS):
    folder.mkdir(parents=True, exist_ok=True)


def run(*command, **kwargs):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(c) for c in command], check=True, **kwargs)


if not REPO.exists():
    run("git", "clone", "-q", "https://github.com/HarshRohila02/nightingale.git", REPO)
run("git", "checkout", "-q", COMMIT, cwd=REPO)
run("git", "log", "--oneline", "-1", cwd=REPO)

# Kaggle's image has most of these; install only what is missing (never on the laptop).
for module, package in (
    ("neo4j", "neo4j"),
    ("dotenv", "python-dotenv"),
    ("networkx", "networkx"),
    ("pyarrow", "pyarrow"),
    ("pydantic", "pydantic"),
    ("yaml", "pyyaml"),
):
    try:
        __import__(module)
    except ImportError:
        run(sys.executable, "-m", "pip", "install", "-q", package)

In [ ]:
# What was added as input: the models, BODHI-S, the retrieval bundle, and an earlier version's results.
sys.path.insert(0, str(REPO / "scripts"))
sys.path.insert(0, str(REPO))
from report_deep import place_models
from run_llm_baselines import find_bundle, place_bodhi_s, previous_results

place_models(INPUT, REPO / "models")
HAS_MODELS = (REPO / "models" / "nightingale_b1_dl").exists()
HAS_BODHI = place_bodhi_s(INPUT, REPO / "data" / "raw" / "bodhi_s")
BUNDLE = find_bundle(INPUT, Path("/tmp/bundle"))
RESUMED = previous_results(INPUT, OUT)
BUNDLE_ARGS = ["--bundle", str(BUNDLE)] if BUNDLE else []
print("models:", "found" if HAS_MODELS else "NOT FOUND: A0+LLM will not run (Add Input -> Your Datasets -> nightingale-models)")
print("BODHI-S:", "found, the graph has its enrichment" if HAS_BODHI else "not found: the graph is DDXPlus and the hand-authored facts only")
print("retrieval bundle:", BUNDLE or "NOT FOUND: B4 and A0+LLM will not run (Add Input -> Notebook Output Files -> the corpus notebook)")
print("results kept from an earlier version:", RESUMED, "line(s)")

In [ ]:
# DDXPlus's validation split, at the snapshot the laptop uses (docs/11 §4). The test split is
# never fetched before Phase 4 (docs/05 §2).
from huggingface_hub import hf_hub_download

for name in ["release_evidences.json", "release_conditions.json", "validate.csv"]:
    hf_hub_download(
        "aai530-group6/ddxplus",
        name,
        repo_type="dataset",
        revision="2ad986acc1ec62fb4a94171acc43f4fdd5bfde53",
        local_dir=str(REPO / "data" / "raw" / "ddxplus"),
    )
run(sys.executable, "scripts/decode_ddxplus.py", cwd=REPO)
run(sys.executable, "scripts/build_ddxplus_chestpain.py", "--split", "validate", cwd=REPO)

In [ ]:
# Ollama, from ollama.com, and the model, from Ollama's registry. The model goes to /tmp: 5 GB
# must not end up in the saved output.
ENV = {**os.environ, "OLLAMA_MODELS": "/tmp/ollama_models"}

try:
    run("nvidia-smi", "-L")
except (OSError, subprocess.CalledProcessError):
    raise SystemExit("No GPU: Session options -> Accelerator -> GPU T4 x2, then Save & Run All again")

# zstd unpacks the download and pciutils lets the installer see the GPU.
run("sh", "-c", "which zstd >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq zstd pciutils)")
pin = f"OLLAMA_VERSION={OLLAMA_VERSION} " if OLLAMA_VERSION else ""
run("sh", "-c", f"curl -fsSL https://ollama.com/install.sh | {pin}sh")

server = subprocess.Popen(
    ["ollama", "serve"], env=ENV, stdout=open(LOGS / "ollama.log", "w"), stderr=subprocess.STDOUT
)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        break
    except OSError:
        time.sleep(2)
else:
    print((LOGS / "ollama.log").read_text()[-3000:])
    raise SystemExit("Ollama did not start; its log is above")

pull = subprocess.run(["ollama", "pull", MODEL], env=ENV, capture_output=True, text=True)
print((pull.stdout + pull.stderr)[-1500:])
if pull.returncode != 0:
    raise SystemExit(f"ollama pull {MODEL} failed")

In [ ]:
# Smoke test: two golden cases and two patients through every arm, first. It fails fast: a model
# that errors, an answer that never parses, a prompt that fills the context, or a model that is
# running on the CPU stops the version here, before the long run. Its figures mean nothing.
RUN = [sys.executable, "scripts/run_llm_baselines.py", "--model", MODEL, "--arms", *ARMS, *BUNDLE_ARGS]
SMOKE = Path("/tmp/llm_smoke")
shutil.rmtree(SMOKE, ignore_errors=True)
subprocess.run([*RUN, "--smoke", "--out", str(SMOKE)], cwd=REPO, env=ENV, check=True)

rows = [json.loads(line) for line in (SMOKE / "results.jsonl").read_text(encoding="utf-8").splitlines()]
seconds = [r["reply"]["elapsed_s"] for r in rows if r.get("reply")]
mean = sum(seconds) / max(len(seconds), 1)
print(f"smoke: {len(rows)} calls, {mean:.1f} s a call on average")
subprocess.run(["ollama", "ps"], env=ENV)  # the PROCESSOR column should say GPU
if mean > 60:
    raise SystemExit("Over a minute a call: Ollama is probably running on the CPU. See logs/ollama.log")

In [ ]:
# The real run. Every call is appended to results.jsonl as it is made, so a run that stops resumes.
# Its log goes to logs/run.log.
started = time.time()
with open(LOGS / "run.log", "a", encoding="utf-8") as handle:
    result = subprocess.run(
        [*RUN, "--out", str(OUT)], cwd=REPO, env=ENV, stdout=handle, stderr=subprocess.STDOUT
    )
minutes = (time.time() - started) / 60
status = "done" if result.returncode == 0 else f"FAILED (exit {result.returncode})"
print(f"run {status} after {minutes:.0f} min")
print((LOGS / "run.log").read_text(encoding="utf-8")[-3000:])

In [ ]:
# Score it, write STATUS.txt, and zip the one file to download (Output tab: llm_explain.zip).
lines = [f"run: {status}, {minutes:.0f} min"]
lines.append("models: " + ("found" if HAS_MODELS else "NOT FOUND, A0+LLM did not run"))
lines.append("retrieval bundle: " + (str(BUNDLE) if BUNDLE else "NOT FOUND, B4 and A0+LLM did not run"))
lines.append("BODHI-S in the graph: " + ("yes" if HAS_BODHI else "no"))
scored = subprocess.run(
    [sys.executable, "scripts/score_llm_baselines.py", str(OUT), "--sheet", str(OUT / "rating_sheet")],
    cwd=REPO, capture_output=True, text=True,
)
print(scored.stdout[-4000:], scored.stderr[-2000:])
if scored.returncode == 0:
    lines += scored.stdout.splitlines()
else:
    lines.append("NOT SCORED: " + scored.stderr[-500:])
finished = result.returncode == 0 and scored.returncode == 0 and (OUT / "results.jsonl").exists()
(OUT / "STATUS.txt").write_text(("DONE" if finished else "NOT DONE") + "\n" + "\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))
server.terminate()
archive = shutil.make_archive("/kaggle/working/llm_explain", "zip", OUT)
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")